In [8]:
!pip install groq python-dotenv

In [22]:
import os
from dotenv import load_dotenv
from groq import Groq
import getpass

# Load .env if exists
load_dotenv()

# Set API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API Key: ")

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])


In [23]:
MODEL_NAME = "llama-3.3-70b-versatile"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a senior technical support engineer.
Be precise, code-focused, logical, and provide debugging steps.
Include example fixes when relevant."""
    },
    "billing": {
        "system_prompt": """You are a billing and subscription support specialist.
Be empathetic, policy-aware, and explain refund or billing steps clearly.
Maintain a professional and calm tone."""
    },
    "general": {
        "system_prompt": """You are a friendly general customer support assistant.
Handle casual conversation and general inquiries helpfully."""
    },
    "tool": {
        "system_prompt": """You are a financial data assistant.
If asked about live crypto prices, you use tools instead of guessing."""
    }
}


In [24]:
def route_prompt(user_input: str) -> str:
    router_prompt = f"""
Classify the following customer message into one of these categories:
[technical, billing, general]

Return ONLY the category name. No explanation.

Message:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": router_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category


In [25]:
def get_bitcoin_price():
    # Mock tool response
    return "The current price of Bitcoin is $64,250 (mock data)."


In [26]:
def process_request(user_input: str) -> str:
    # Tool routing (Bonus)
    if "bitcoin" in user_input.lower() and "price" in user_input.lower():
        return get_bitcoin_price()

    # Step 1: Route
    category = route_prompt(user_input)

    # Fallback safety
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 2: Call Expert
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content


In [27]:
# Technical Test
print(process_request("My python script is throwing an IndexError on line 5."))

# Billing Test
print(process_request("I was charged twice for my subscription this month."))

# General Test
print(process_request("What are your business hours?"))

# Tool Test
print(process_request("What is the current price of Bitcoin?"))


To assist you with the `IndexError` on line 5 of your Python script, I'll need more information. However, I can guide you through a general troubleshooting process.

### Understanding the Error
An `IndexError` typically occurs when you try to access an element in a list (or other sequence type) using an index that is out of range. For example, if you have a list with 5 elements (indices 0 through 4), trying to access the element at index 5 would raise an `IndexError`.

### Troubleshooting Steps
1. **Review Line 5**: Look at line 5 of your script and identify the line of code causing the error. This line likely involves accessing an element from a list, tuple, or string.

2. **Check Indexing**: Verify that the index you're using is within the bounds of the sequence you're trying to access. Remember, indexing starts at 0, so the last valid index of a sequence is its length minus 1.

3. **Sequence Length**: Before accessing an element, ensure you know the length of your sequence. You can 